# Notes:
1. the chain IDs in `proteins_SI_90.csv` are not valid (non-existing chains are occuring). We had to recalculate them (`full_scPDB.csv`) using the IFP.txt files in the scPDB dataset.

In [1]:
import pandas as pd
df = pd.read_csv('/home/skrhakv/CryptoBench/data/A-filter-ahojdb/all_apoholo_all_ligands/pairs.csv')


/tmp/slurm.148767/ipykernel_1777366/118256366.py:2: DtypeWarning: Columns (29,30,34,35,70,82,83,87,88) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/skrhakv/CryptoBench/data/A-filter-ahojdb/all_apoholo_all_ligands/pairs.csv')


In [ ]:
import csv   
proteins = []
proteins_without_chains = []
scPDB_binding_sites = {}
with open('/home/skrhakv/cryptoshow-analysis/data/E-regular-binding-site-predictor/full_scPDB.csv', 'r') as f:
    csv_file = csv.reader(f, delimiter=';')
    for row in csv_file:
        proteins.append(f'{row[0]}{row[1]}')
        proteins_without_chains.append(row[0])
        scPDB_binding_sites[f'{row[0]}{row[1]}'] = row[3].split(' ')
scPDB_subset = df[df['apo_structure'].isin(proteins_without_chains) | df['holo_structure'].isin(proteins_without_chains)]


In [ ]:
def extract_residues(pocket_selection):
    residues_chunk = pocket_selection.split(' ')[-2][:-1]
    return residues_chunk.split('+')

binding_sites = {}
for protein_id in proteins:
    print(f'Processing {protein_id}...')
    binding_sites[protein_id] = set()
    pdb_id, chain_id = protein_id[:4], protein_id[4:]
    apo_subset = scPDB_subset[(scPDB_subset['apo_structure'] == pdb_id) & (scPDB_subset['apo_chains3'] == chain_id)]
    
    if not apo_subset.empty:
        for pocket_selection in apo_subset['apo_pocket_selection']:
            binding_sites[protein_id].update(extract_residues(pocket_selection))
    holo_subset = scPDB_subset[(scPDB_subset['holo_structure'] == pdb_id) & (scPDB_subset['holo_chains3'] == chain_id)]

    if not holo_subset.empty:
        for pocket_selection in holo_subset['holo_pocket_selection']:
            binding_sites[protein_id].update(extract_residues(pocket_selection))

import pickle
with open('/home/skrhakv/CryptoBench/src/A-filter-ahojdb/binding_sites.pkl', 'wb') as f:
    pickle.dump(binding_sites)

# Merge
Here we are merging the scPDB and additional residues from AhojDB. We are safe because scPDB uses auth labels (see 1uh5 as an example), and AhojDB uses the same.

In [11]:
binding_sites_enhanced = {}
for protein_id in proteins:
    binding_sites_enhanced[protein_id] = binding_sites[protein_id].union(set(scPDB_binding_sites[protein_id]))

with open('/home/skrhakv/cryptoshow-analysis/data/E-regular-binding-site-predictor/scPDB_enhanced_binding_sites.csv', 'w') as f:
    for protein_id in proteins:
        f.write(f'{protein_id[:4]};{protein_id[4:]};NON_CRYPTIC;{" ".join(sorted(binding_sites_enhanced[protein_id]))};UNKNOWN\n')